# 06 · Functions, recursion, and LeetCode's classes

**How this notebook works**
- Run the setup cell below first. Then go top to bottom, running each cell with **Shift+Enter**.
- Gray **example** cells: run them, then change something and run again.
- **Try it** cells: your scratch space for each section.
- **Predict the output**: write your guess in the comment *before* running.
- **Exercises** at the bottom: write your code, run the cell, then run the `check(...)` cell under it.

In [ ]:
# ▶ Run this cell first (Shift+Enter). It loads the test helper and the same imports LeetCode gives you.
import sys, pathlib
sys.path.insert(0, str(next(p for p in (pathlib.Path.cwd() / "_primer", pathlib.Path.cwd() / "python-primer" / "_primer") if p.exists())))
from check import check, summary, lesson
lesson("ex06_functions_recursion")

import bisect, heapq, math, random
from collections import Counter, OrderedDict, defaultdict, deque
from functools import cache, lru_cache
from itertools import accumulate, combinations, groupby, pairwise, permutations, product
from typing import List, Optional

**Warm-up (5 min):** before reading, close everything and write down every trap you remember from lessons 01–05. Then check them against your *My traps* notes in those notebooks.

## What a LeetCode Python problem looks like

In [ ]:
from typing import Optional

class TreeNode:                     # LeetCode defines this for you
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

class Solution:
    def countNodes(self, root: Optional[TreeNode]) -> int:
        if not root:
            return 0
        return 1 + self.countNodes(root.left) + self.countNodes(root.right)

root = TreeNode(1, TreeNode(2), TreeNode(3, TreeNode(4)))
print(Solution().countNodes(root))  # LeetCode makes this call for you

- `self` is required as the first parameter of every method. You never pass it yourself.
- `Optional[TreeNode]` means "a TreeNode or None". Type hints are documentation only.
- To call another method you wrote: `self.helper(x)`. Most people use a nested function instead (below).

The node classes LeetCode gives you:

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

`__init__` is the constructor; `self.x = ...` creates fields. Build a node with `ListNode(3)`, no `new`.

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Linked list moves

In [ ]:
head = ListNode(1, ListNode(2, ListNode(3)))
dummy = ListNode(0, head)      # dummy node: removes the "what if I delete the head" special case
cur = head
while cur:                     # stop after the last node
    print(cur.val, end=" ")
    cur = cur.next
print()
cur = head
while cur and cur.next:        # stop at the last node (or for fast/slow pointers)
    cur = cur.next
print("last node:", cur.val)

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Nested functions and `nonlocal`

The usual shape of a DFS:

In [ ]:
class Solution:
    def diameterOfBinaryTree(self, root):
        best = 0

        def depth(node):
            nonlocal best                     # required because we ASSIGN to best below
            if not node:
                return 0
            l, r = depth(node.left), depth(node.right)
            best = max(best, l + r)
            return 1 + max(l, r)

        depth(root)
        return best

root = TreeNode(1, TreeNode(2, TreeNode(4), TreeNode(5)), TreeNode(3))
print(Solution().diameterOfBinaryTree(root))  # 3: the path 4-2-1-3

Rules for the inner function:
- **Reading** an outer variable: works.
- **Mutating** an outer list/dict/set (`res.append(x)`, `seen.add(x)`): works.
- **Reassigning** an outer variable (`best = ...`, `count += 1`): needs `nonlocal best`, or you get `UnboundLocalError`.

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Recursion limits

Python's default recursion limit is about 1000 frames. A linked list or a skewed tree with 10⁴ nodes will crash a recursive solution with `RecursionError`. Options: rewrite iteratively with an explicit stack (best), or `sys.setrecursionlimit(10**5)` (works on LeetCode; mention the trade-off in an interview).

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Memoization with @cache

In [ ]:
from functools import cache

def unique_paths(m, n):
    @cache
    def paths(r, c):              # ways to reach (r, c) moving only right or down
        if r == 0 or c == 0:
            return 1
        return paths(r - 1, c) + paths(r, c - 1)
    return paths(m - 1, n - 1)

print(unique_paths(3, 7))         # 28
print(unique_paths(18, 18))       # instant with @cache; would take forever without it

`@cache` remembers results per argument tuple, which turns exponential recursion into top-down DP. Arguments must be hashable (ints, strings, tuples; not lists). `@lru_cache(None)` is the same thing on older Python. Define the cached function **inside** the method so the cache doesn't leak between test cases.

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Backtracking skeleton

In [ ]:
nums = [1, 2, 3]
res, path = [], []

def backtrack(start):
    res.append(path[:])               # snapshot, not path itself
    for i in range(start, len(nums)):
        path.append(nums[i])          # choose
        backtrack(i + 1)              # explore
        path.pop()                    # un-choose

backtrack(0)
print(res)                            # all 8 subsets

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Writing your own class (design problems)

Several Top 150 problems hand you an empty class to implement: Min Stack, Implement Trie, LRU Cache, Insert Delete GetRandom O(1). LeetCode creates your object and calls the methods; you pick the fields.

In [ ]:
class MinStack:
    def __init__(self):
        self.stack = []          # fields live on self
        self.mins = []           # mins[i] = min of stack[0..i]

    def push(self, val: int) -> None:
        self.stack.append(val)
        self.mins.append(min(val, self.mins[-1]) if self.mins else val)

    def getMin(self) -> int:
        return self.mins[-1]

Forgetting `self.` is the usual bug: `stack.append(val)` inside a method looks for a local `stack` and raises `NameError`.

**Trie:** a node is just a dict of children plus an end flag.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}       # char -> TrieNode
        self.end = False

root = TrieNode()
for word in ["cat", "car"]:
    node = root
    for c in word:
        if c not in node.children:
            node.children[c] = TrieNode()
        node = node.children[c]
    node.end = True
print(list(root.children["c"].children["a"].children))   # ['t', 'r']

**LRU Cache:** `OrderedDict` is a dict that remembers order and can move keys around in O(1).

In [ ]:
from collections import OrderedDict
lru = OrderedDict()
for k, v in [("a", 1), ("b", 2), ("c", 3)]:
    lru[k] = v
lru.move_to_end("a")          # mark "a" as most recently used
print(list(lru))              # ['b', 'c', 'a']
lru.popitem(last=False)       # evict the least recently used (the front)
print(list(lru))              # ['c', 'a']

Interviewers sometimes ask for LRU Cache *without* OrderedDict, meaning a hash map plus your own doubly linked list. Know that `OrderedDict` exists, and be ready to explain why the hand-built version is O(1).

**Random:** `import random`, then `random.choice(lst)` picks a uniform random element in O(1) (lists only, not sets) and `random.randint(a, b)` is inclusive on both ends.

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Lambdas and the mutable default trap

`key=lambda x: x[1]` is an inline one-expression function. Use it for sort keys; don't build logic in it.

In [ ]:
def add_wrong(x, acc=[]):     # WRONG: the same list is reused across every call
    acc.append(x)
    return acc

def add_right(x, acc=None):   # RIGHT
    if acc is None:
        acc = []
    acc.append(x)
    return acc

print(add_wrong(1), add_wrong(2))
print(add_right(1), add_right(2))

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Predict the output

For each one: type your guess into the comment line, **then** run the cell. The output is the answer. If you were wrong, figure out why before moving on.

**1.** What does this print?

In [ ]:
# My guess: 
def outer():
    count = 0
    def inc():
        count += 1
    inc()
    return count
outer()

**2.** What does this print?

In [ ]:
# My guess: 
def add(x, acc=[]):
    acc.append(x)
    return acc
add(1); print(add(2))

**3.** What does this print?

In [ ]:
# My guess: 
res, path = [], [1]
res.append(path)
path.append(2)
print(res)

<details><summary>Why (open after running all of them)</summary>

1. `UnboundLocalError` (assigning `count` makes it local to `inc`; needs `nonlocal count`)
2. `[1, 2]`
3. `[[1, 2]]`
</details>

---
# Exercises

Write your code in each cell (replace `raise NotImplementedError`), run it, then run the `check(...)` cell below it. `ok` means done; `FAIL` shows what was expected; `TODO` means not started.

**Given:** LeetCode defines `ListNode` for you. Run this cell, don't change it.

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

**Given:** LeetCode defines `TreeNode` for you. Run this cell, don't change it.

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

### 1. `build_list`

Build a linked list from a Python list and return the head (None if empty). Use a dummy node.

```
build_list([1, 2, 3])  ->  head of 1 → 2 → 3
build_list([])         ->  None
```

In [ ]:
def build_list(values: list) -> Optional[ListNode]:
    raise NotImplementedError

### 2. `to_pylist`

Linked list -> Python list.

```
to_pylist(1 → 2 → 3)  ->  [1, 2, 3]
to_pylist(None)       ->  []
```

In [ ]:
def to_pylist(head: Optional[ListNode]) -> list:
    raise NotImplementedError

In [ ]:
check("build_and_to_pylist")

On LeetCode, your code goes inside `class Solution`. Each cell below defines `Solution` with just one method so you can test them one at a time. That's fine; the test right below each cell uses the version you just ran.

### 3. `reverseList` · LeetCode 206

Solve it, iteratively, O(1) space.

```
reverseList(1 → 2 → 3 → 4)  ->  4 → 3 → 2 → 1
reverseList(None)           ->  None
```

In [ ]:
class Solution:
    def reverseList(self, head: Optional[ListNode]) -> Optional[ListNode]:
        raise NotImplementedError

In [ ]:
check("reverse_list")

### 4. `removeElements` · LeetCode 203

Remove every node with node.val == val. Use a dummy node.

```
removeElements(7 → 1 → 7 → 2 → 7, val=7)  ->  1 → 2
removeElements(7 → 7, val=7)              ->  None
```

In [ ]:
class Solution:
    def removeElements(self, head: Optional[ListNode], val: int) -> Optional[ListNode]:
        raise NotImplementedError

In [ ]:
check("remove_elements")

### 5. `maxDepth` · LeetCode 104

Solve it, recursively.

```
     3
    / \
   9   20
      /  \
     15   7

maxDepth(root of the tree above)  ->  3
maxDepth(None)                    ->  0
```

In [ ]:
class Solution:
    def maxDepth(self, root: Optional[TreeNode]) -> int:
        raise NotImplementedError

In [ ]:
check("max_depth")

### 6. `countLeaves`

Count leaves with a nested dfs() that increments an outer counter (needs nonlocal).

```
countLeaves(the tree from maxDepth)  ->  3      (9, 15 and 7)
countLeaves(None)                    ->  0
```

In [ ]:
class Solution:
    def countLeaves(self, root: Optional[TreeNode]) -> int:
        raise NotImplementedError

In [ ]:
check("count_leaves")

### 7. `climbStairs` · LeetCode 70

Solve it with a nested @cache function. Must handle n = 80 instantly.

```
climbStairs(2)   ->  2
climbStairs(5)   ->  8
climbStairs(80)  ->  37889062373143906
```

In [ ]:
class Solution:
    def climbStairs(self, n: int) -> int:
        raise NotImplementedError

In [ ]:
check("climb_stairs")

### 8. `combine` · LeetCode 77

All combinations of k numbers from 1..n, in lexicographic order. Backtracking with path.append / recurse / path.pop.

```
combine(4, 2)  ->  [[1,2], [1,3], [1,4], [2,3], [2,4], [3,4]]
combine(1, 1)  ->  [[1]]
```

In [ ]:
class Solution:
    def combine(self, n: int, k: int) -> List[List[int]]:
        raise NotImplementedError

In [ ]:
check("combine")

### 9. `MinStack` (class) · LeetCode 155

Push, pop, top, getMin, all O(1).

```
s = MinStack()
s.push(-2); s.push(0); s.push(-3)
s.getMin()  ->  -3
s.pop()
s.top()     ->  0
s.getMin()  ->  -2
```

In [ ]:
class MinStack:

    def __init__(self):
        raise NotImplementedError

    def push(self, val: int) -> None:
        raise NotImplementedError

    def pop(self) -> None:
        raise NotImplementedError

    def top(self) -> int:
        raise NotImplementedError

    def getMin(self) -> int:
        raise NotImplementedError

In [ ]:
check("min_stack")

### 10. `Trie` (class) · LeetCode 208

Insert, search (whole word), startsWith (prefix). Write a small TrieNode class (or use nested dicts).

```
t = Trie()
t.insert("apple")
t.search("apple")    ->  True
t.search("app")      ->  False   (only a prefix)
t.startsWith("app")  ->  True
```

In [ ]:
class Trie:

    def __init__(self):
        raise NotImplementedError

    def insert(self, word: str) -> None:
        raise NotImplementedError

    def search(self, word: str) -> bool:
        raise NotImplementedError

    def startsWith(self, prefix: str) -> bool:
        raise NotImplementedError

In [ ]:
check("trie")

### 11. `LRUCache` (class) · LeetCode 146

Solve it using collections.OrderedDict. get returns -1 if missing. Both get and put count as a use; put evicts the least recently used key when over capacity.

```
c = LRUCache(2)
c.put(1, 1); c.put(2, 2)
c.get(1)  ->  1       (1 is now the most recently used)
c.put(3, 3)            (over capacity: evicts 2)
c.get(2)  ->  -1
```

In [ ]:
class LRUCache:

    def __init__(self, capacity: int):
        raise NotImplementedError

    def get(self, key: int) -> int:
        raise NotImplementedError

    def put(self, key: int, value: int) -> None:
        raise NotImplementedError

In [ ]:
check("lru_cache")

---
## Wrap-up

In [ ]:
summary()

## My traps

*Double-click to edit.* Write the 3–5 things from this lesson that surprised you or that you got wrong, in your own words. You'll use these in the next warm-up and the exit test.

1. 
2. 
3. 

**Done?** All `ok` in the summary → commit and sync → tell Claude **"lesson 06 done."**